In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ===================== USER SETTINGS =====================
MODE = "batch"      
# Options:
# "single" -> one CSV file
# "batch"  -> all CSV files in a folder

SINGLE_FILE_PATH = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/drill_mech_depth_30jul20.csv"
SINGLE_DESC_PATH = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/desc/drill_mech_depth_30jul20_curve_summary.txt"

BATCH_DESC_FOLDER = "/Users/apple/Downloads/welldata/2020/ROTTERDAM-08-SIDETRACK1/structured/desc/"
BATCH_FOLDER_PATH = "/Users/apple/Downloads/welldata/2020/ROTTERDAM-08-SIDETRACK1/structured/csv/"   # change if needed

OUTPUT_FOLDER = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
# ==========================================================


DUMMY_VALUE = -999.25

# --------- MANUAL SCALE CONTROL ----------
MANUAL_X_SCALE = {
    # Example:
    # "GR": (0, 150),
    # "RHOB": (1.9, 2.9),
}
# -----------------------------------------


def get_auto_scale(series, dummy=DUMMY_VALUE, pad_fraction=0.08):
    # Mask dummy only for scaling
    clean = series.replace(dummy, np.nan).dropna()

    if len(clean) == 0:
        return (0, 1)

    vmin = clean.min()
    vmax = clean.max()

    span = vmax - vmin
    if span == 0:
        span = abs(vmin) if vmin != 0 else 1

    pad = span * pad_fraction
    return float(vmin - pad), float(vmax + pad)


def load_curve_descriptions(desc_file_path):
    """
    Reads *_curve_summary.txt and returns:
    {MNEM: (UNIT, DESCRIPTION)}

    Correctly handles units with spaces like '1000 kgf', 'ft.lbf', etc.
    """
    descriptions = {}

    if not os.path.exists(desc_file_path):
        return descriptions

    with open(desc_file_path, "r") as f:
        for line in f:
            line = line.rstrip()

            if not line or line.startswith("~") or line.startswith("#"):
                continue

            # Format:
            # MNEM<spaces>.<UNIT><spaces><DESCRIPTION>

            if "." not in line:
                continue

            mnem = line[:15].strip()

            rest = line[15:].lstrip()
            if not rest.startswith("."):
                continue

            rest = rest[1:]  # remove leading dot

            # Split UNIT and DESCRIPTION by 2+ spaces
            parts = rest.split("  ", 1)

            if len(parts) != 2:
                continue

            unit = parts[0].strip()
            desc = parts[1].strip()

            descriptions[mnem] = (unit, desc)

    return descriptions


def plot_well_log(csv_path, save_folder, desc_file_path=None):

    CURVE_DESCRIPTIONS = load_curve_descriptions(desc_file_path)

    df = pd.read_csv(csv_path)
    df = df.apply(pd.to_numeric, errors='coerce')
    #df.replace(DUMMY_VALUE, np.nan, inplace=True)
    df.dropna(how="all", inplace=True)

    y = df.iloc[:, 0]
    features = list(df.columns[1:])
    n_tracks = len(features)

    tracks_per_image = 13
    n_images = int(np.ceil(n_tracks / tracks_per_image))

    base = os.path.splitext(os.path.basename(csv_path))[0]

    # ---- If more than 13 curves → create subfolder ----
    if n_images > 1:
        plot_folder = os.path.join(save_folder, base)
        os.makedirs(plot_folder, exist_ok=True)
    else:
        plot_folder = save_folder

    for img_idx in range(n_images):

        start = img_idx * tracks_per_image
        end = min(start + tracks_per_image, n_tracks)
        subset_features = features[start:end]

        fig, axes = plt.subplots(
            nrows=1,
            ncols=len(subset_features),
            figsize=(3*len(subset_features), 14),
            sharey=True
        )

        if len(subset_features) == 1:
            axes = [axes]

        scale_report = {}

        for i, feature in enumerate(subset_features):
            ax = axes[i]
            curve = df[feature]

            ax.plot(curve, y, linewidth=0.8)

            # ---- Scaling ----
            if feature in MANUAL_X_SCALE:
                xmin, xmax = MANUAL_X_SCALE[feature]
                scale_type = "MANUAL"
            else:
                xmin, xmax = get_auto_scale(curve)
                scale_type = "AUTO"

            ax.set_xlim(xmin, xmax)
            ax.margins(x=0)
            ax.set_xlabel(feature)
            ax.grid(True)
            ax.invert_yaxis()

            scale_report[feature] = (scale_type, xmin, xmax)

            # ---- Description under track ----
            # ---- X-axis label with unit ----
            if feature in CURVE_DESCRIPTIONS:
                unit, description = CURVE_DESCRIPTIONS[feature]
                ax.set_xlabel(f"{feature} ({unit})")
            else:
                ax.set_xlabel(feature)

            # ---- Description under track ----
            if feature in CURVE_DESCRIPTIONS:
                text = CURVE_DESCRIPTIONS[feature][1]   # DESCRIPTION only
            else:
                text = "No description available"

            # ---- Draw description ----
            ax.text(0.5, -0.08, text,
                    ha="center", va="top",
                    transform=ax.transAxes,
                    fontsize=8, wrap=True)



        axes[0].set_ylabel(df.columns[0])

        # ---- Title with included logs ----
        log_list = ", ".join(subset_features)
        fig.suptitle(
            f"{base}.csv   |   Logs included: {log_list}",
            fontsize=13
        )

        # ---- Save image ----
        if n_images > 1:
            out_name = f"{base}_part{img_idx+1}.png"
        else:
            out_name = f"{base}_welllog.png"

        out_path = os.path.join(plot_folder, out_name)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(out_path, dpi=300)
        plt.close()

        print("Saved:", out_path)


# ===================== RUN =====================
if MODE == "single":
    plot_well_log(
        SINGLE_FILE_PATH,
        OUTPUT_FOLDER,
        desc_file_path=SINGLE_DESC_PATH
    )

elif MODE == "batch":
    for file in os.listdir(BATCH_FOLDER_PATH):
        if file.lower().endswith(".csv"):
            csv_path = os.path.join(BATCH_FOLDER_PATH, file)

            # matching description file
            base = os.path.splitext(file)[0]
            desc_path = os.path.join(BATCH_DESC_FOLDER, base + "_curve_summary.txt")

            plot_well_log(
                csv_path,
                OUTPUT_FOLDER,
                desc_file_path=desc_path
            )

print("All done.")


/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_76313/2431727989.py:103: DtypeWarning: Columns (0,1,2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_time_01_07oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_03_06oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_03_07oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/drill_mech_depth_30jul20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_08oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/drill_mech_time_30jul20/drill_mech_time_30jul20_part1.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/drill_mech_time_30jul20/drill_mech_time_30jul20_part2.png


/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_76313/2431727989.py:103: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250

Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part1.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part2.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part3.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part4.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part5.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part6.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part7.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_01_07oct20/snl_01_07oct20_part8.png
Saved: /Users/apple/Downloads/welldata/R

/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_76313/2431727989.py:103: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250

Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part1.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part2.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part3.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part4.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part5.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part6.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part7.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_02_07oct20/snl_02_07oct20_part8.png
Saved: /Users/apple/Downloads/welldata/R

/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_76313/2431727989.py:103: DtypeWarning: Columns (0,2,3,4,5,6,8,9,10,11,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/drill_mech_time_03aug20/drill_mech_time_03aug20_part1.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/drill_mech_time_03aug20/drill_mech_time_03aug20_part2.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_02_07oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_02_06oct20_welllog.png


/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_76313/2431727989.py:103: DtypeWarning: Columns (0,1,2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_time_03_07oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_01_07oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_01_06oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/lwd_03aug20/lwd_03aug20_part1.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/lwd_03aug20/lwd_03aug20_part2.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/lwd_03aug20/lwd_03aug20_part3.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_08_07oct20_welllog.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_ccl_11_07oct20/snl_ccl_11_07oct20_part1.png
Saved: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/plots/snl_c